# Dynamic Waterfall Example

This notebook walks through the same scenario as `examples/scripts/statements/dynamic_waterfall_example.py`, but interactively. We will:

1. Build an operating model with revenue, EBITDA, CapEx, taxes, and liquidity inputs.
2. Add a capital structure with a term loan, an Excess Cash Flow (ECF) sweep, and a liquidity-driven PIK toggle.
3. Evaluate the model with a simple market context and inspect the resulting cash waterfall.

> The example focuses on API usage. For production valuation you must supply calibrated market data and instrument specs that match your internal schemas exactly.


In [14]:
from datetime import date

import pandas as pd

from finstack.core.dates.periods import PeriodId
from finstack.core.market_data import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.statements.builder import ModelBuilder
from finstack.statements.evaluator import EvaluatorWithContext
from finstack.statements.types import (
    AmountOrScalar,
    EcfSweepSpec,
    PaymentPriority,
    PikToggleSpec,
    WaterfallSpec,
)

pd.set_option("display.float_format", "{:.0f}".format)


## 1. Build the operating model

We configure the core forecast nodes (revenue, EBITDA, CapEx, taxes) plus a liquidity metric that will later control the PIK toggle.


In [ ]:
builder = ModelBuilder.new("LBO Model")
builder.periods("2025Q1..2026Q4", None)

builder.value(
    "revenue",
    [
        (PeriodId.quarter(2025, 1), AmountOrScalar.scalar(1_000_000.0)),
        (PeriodId.quarter(2025, 2), AmountOrScalar.scalar(2_000_000.0)),  # High revenue
        (PeriodId.quarter(2025, 3), AmountOrScalar.scalar(1_000_000.0)),
        (PeriodId.quarter(2025, 4), AmountOrScalar.scalar(1_000_000.0)),
        (PeriodId.quarter(2026, 1), AmountOrScalar.scalar(1_000_000.0)),
        (PeriodId.quarter(2026, 2), AmountOrScalar.scalar(1_000_000.0)),
        (PeriodId.quarter(2026, 3), AmountOrScalar.scalar(1_000_000.0)),
        (PeriodId.quarter(2026, 4), AmountOrScalar.scalar(1_000_000.0)),
    ],
)

builder.compute("ebitda", "revenue * 0.4")  # 40% EBITDA margin
builder.compute("capex", "revenue * 0.05")  # 5% CapEx
builder.compute("taxes", "ebitda * 0.25")   # 25% tax rate

builder.value(
    "liquidity",
    [
        (PeriodId.quarter(2025, 1), AmountOrScalar.scalar(50_000.0)),  # Triggers PIK (< 100k)
        (PeriodId.quarter(2025, 2), AmountOrScalar.scalar(500_000.0)),
        (PeriodId.quarter(2025, 3), AmountOrScalar.scalar(200_000.0)),
        (PeriodId.quarter(2025, 4), AmountOrScalar.scalar(250_000.0)),
        (PeriodId.quarter(2026, 1), AmountOrScalar.scalar(300_000.0)),
        (PeriodId.quarter(2026, 2), AmountOrScalar.scalar(320_000.0)),
        (PeriodId.quarter(2026, 3), AmountOrScalar.scalar(340_000.0)),
        (PeriodId.quarter(2026, 4), AmountOrScalar.scalar(360_000.0)),
    ],
)

builder


## 2. Add the capital structure

We load a $10MM term loan spec (mirroring the JSON handed to `builder.add_custom_debt`) and keep all optional fields explicit because the Rust serializers deny unknown fields.


In [ ]:
term_loan_spec = {
    "id": "TL-A",
    "currency": "USD",
    "notional_limit": {"amount": "10000000", "currency": "USD"},
    "issue": "2024-12-31",
    "maturity": "2030-01-01",
    "rate": {"Fixed": {"rate_bp": 800}},
    "pay_freq": {"Months": 3},
    "day_count": "Act360",
    "bdc": "modified_following",
    "calendar_id": None,
    "stub": "None",
    "discount_curve_id": "USD-OIS",
    "credit_curve_id": None,
    "amortization": "None",
    "coupon_type": "Cash",
    "upfront_fee": None,
    "ddtl": {
        "commitment_limit": {"amount": "10000000", "currency": "USD"},
        "availability_start": "2024-12-31",
        "availability_end": "2026-01-01",
        "draws": [
            {
                "date": "2024-12-31",
                "amount": {"amount": "10000000", "currency": "USD"},
            }
        ],
        "commitment_step_downs": [],
        "usage_fee_bp": 0,
        "commitment_fee_bp": 0,
        "fee_base": "Undrawn",
        "oid_policy": None,
    },
    "covenants": None,
    "pricing_overrides": {
        "quoted_clean_price": None,
        "rho_bump_decimal": None,
        "vega_bump_decimal": None,
        "implied_volatility": None,
        "quoted_spread_bp": None,
        "upfront_payment": None,
        "ytm_bump_decimal": None,
        "theta_period": None,
        "mc_seed_scenario": None,
        "adaptive_bumps": False,
        "spot_bump_pct": None,
        "vol_bump_pct": None,
        "rate_bump_bp": None,
    },
    "call_schedule": None,
    "attributes": {"tags": [], "meta": {}},
}

builder.add_custom_debt("TL-A", term_loan_spec)

term_loan_spec


{'id': 'TL-A',
 'currency': 'USD',
 'notional_limit': {'amount': '10000000', 'currency': 'USD'},
 'issue': '2024-12-31',
 'maturity': '2030-01-01',
 'rate': {'Fixed': {'rate_bp': 800}},
 'pay_freq': {'Months': 3},
 'day_count': 'Act360',
 'bdc': 'modified_following',
 'calendar_id': None,
 'stub': 'None',
 'discount_curve_id': 'USD-OIS',
 'credit_curve_id': None,
 'amortization': 'None',
 'coupon_type': 'Cash',
 'upfront_fee': None,
 'ddtl': {'commitment_limit': {'amount': '10000000', 'currency': 'USD'},
  'availability_start': '2024-12-31',
  'availability_end': '2026-01-01',
  'draws': [{'date': '2024-12-31',
    'amount': {'amount': '10000000', 'currency': 'USD'}}],
  'commitment_step_downs': [],
  'usage_fee_bp': 0,
  'commitment_fee_bp': 0,
  'fee_base': 'Undrawn',
  'oid_policy': None},
 'covenants': None,
 'pricing_overrides': {'quoted_clean_price': None,
  'rho_bump_decimal': None,
  'vega_bump_decimal': None,
  'implied_volatility': None,
  'quoted_spread_bp': None,
  'upfront

## 3. Configure the waterfall mechanics

We assign a priority of payments, a 50% ECF sweep, and a liquidity-driven PIK toggle that flips the term loan from cash to PIK interest when liquidity drops below $100k.


In [ ]:
waterfall = WaterfallSpec(
    priority_of_payments=[
        PaymentPriority.Fees,
        PaymentPriority.Interest,
        PaymentPriority.Amortization,
        PaymentPriority.Sweep,
        PaymentPriority.Equity,
    ],
    ecf_sweep=EcfSweepSpec(
        ebitda_node="ebitda",
        sweep_percentage=0.5,
        taxes_node="taxes",
        capex_node="capex",
        target_instrument_id="TL-A",
    ),
    pik_toggle=PikToggleSpec(
        liquidity_metric="liquidity",
        threshold=100_000.0,
        target_instrument_ids=["TL-A"],
    ),
)

builder.waterfall(waterfall)

builder.compute("interest_expense", "cs.interest_expense.total")
builder.compute("interest_cash", "cs.interest_expense_cash.total")
builder.compute("interest_pik", "cs.interest_expense_pik.total")
builder.compute("principal_payment", "cs.principal_payment.total")
builder.compute("debt_balance", "cs.debt_balance.total")

waterfall


WaterfallSpec(priority=5, ecf=true, pik=true)

## 4. Build the model and evaluate with a simple market context

Capital structure evaluation requires market data. For illustration we insert a flat USD OIS discount curve and run the evaluator as of 2025-01-01.


In [ ]:
model = builder.build()
as_of = date(2025, 1, 1)

market_ctx = MarketContext()
market_ctx.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (1.0, 0.96), (5.0, 0.8), (10.0, 0.6)],
    )
)

evaluator = EvaluatorWithContext.new(market_ctx, as_of)
results = evaluator.evaluate(model)

model, results


(FinancialModelSpec(id='LBO Model', periods=8, nodes=10),
 Results(nodes=10, periods=8))

## 5. Inspect the waterfall outputs

We collect the key nodes into a DataFrame and highlight the PIK toggle (Q1) and the ECF sweep (Q2).


In [19]:
rows = []
for period in model.periods:
    pid = period.id
    rows.append(
        {
            "Period": str(pid),
            "EBITDA": results.get("ebitda", pid),
            "Liquidity": results.get_or("liquidity", pid, 0.0),
            "Interest": results.get("interest_expense", pid),
            "Principal": results.get("principal_payment", pid),
            "Debt Balance": results.get("debt_balance", pid),
            "Interest (Cash)": results.get("interest_cash", pid),
            "Interest (PIK)": results.get_or("interest_pik", pid, 0.0),
        }
    )

df = pd.DataFrame(rows)
df


,Period,EBITDA,Liquidity,Interest,Principal,Debt Balance,Interest (Cash),Interest (PIK)
0,2025Q1,400000,50000,200000,0,200000,0,200000
1,2025Q2,800000,500000,202222,200000,0,202222,0
2,2025Q3,400000,200000,204444,0,0,204444,0
3,2025Q4,400000,250000,202222,0,0,202222,0
4,2026Q1,400000,300000,200000,0,0,200000,0
5,2026Q2,400000,320000,204444,0,0,204444,0
6,2026Q3,400000,340000,204444,0,0,204444,0
7,2026Q4,400000,360000,202222,0,0,202222,0


### Toggle checks

These quick checks mirror the script's console output.


In [ ]:
q1_pik = results.get_or("interest_pik", PeriodId.quarter(2025, 1), 0.0)
q2_principal = results.get("principal_payment", PeriodId.quarter(2025, 2))

print("Q1 PIK Toggle:", "Active" if q1_pik > 0 else "Inactive", f"({q1_pik:,.0f})")
print("Q2 ECF Sweep:", "Active" if q2_principal > 0 else "Inactive", f"({q2_principal:,.0f})")


Q1 PIK Toggle: Active (200,000)
Q2 ECF Sweep: Active (200,000)
